# Lesson 13: Exporting Scikit-Learn to ONNX & Building a Cuisine Recommender

Train a 380-feature `SVC(kernel='linear', probability=True)` classifier, export it to `model.onnx` using `skl2onnx`, and run inference in Python with `onnxruntime`.

In [ ]:
!pip install -q skl2onnx onnxruntime scikit-learn

In [ ]:
import numpy as np
import onnxruntime as rt
from sklearn.svm import SVC
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

rng = np.random.default_rng(13)
X = np.zeros((250, 380), dtype=np.float32)
y = np.repeat(['indian', 'japanese', 'korean', 'chinese', 'thai'], 50)
# Signal indices matching lesson: cumin=112, fenugreek=126, sake=302, soy_sauce=327
X[0:50, [112, 126]] = 1.0
X[50:100, [302, 327]] = 1.0
X[100:150, [20, 327]] = 1.0
X[150:200, [50, 327]] = 1.0
X[200:250, [80, 112]] = 1.0
X += rng.binomial(1, 0.02, size=X.shape).astype(np.float32)

model = SVC(kernel='linear', C=10, probability=True, random_state=0).fit(X, y)
initial_type = [('float_input', FloatTensorType([None, 380]))]
options = {id(model): {'nocl': True, 'zipmap': False}}
onx = convert_sklearn(model, initial_types=initial_type, options=options)
with open('model.onnx', 'wb') as f:
    f.write(onx.SerializeToString())

sess = rt.InferenceSession('model.onnx')
sample = np.zeros((1, 380), dtype=np.float32)
sample[0, [112, 126]] = 1.0
pred = sess.run(None, {'float_input': sample})[0]
print('ONNX Runtime Prediction for [cumin, fenugreek]:', pred[0])

## Assignment: Build a New Recommender

Test `sample[0, [302, 327]] = 1.0` (`sake` + `soy_sauce`) through `sess.run(...)` and inspect the predicted probabilities.